# HierarchicalDet Phase 0 setup (Kaggle GPU notebook)

Scope: environment install, DENTEX dataset download + 3-tier verification, and a single-image forward-pass smoke test of the inference pipeline (no pretrained HierarchicalDet weights are publicly released, so this only proves the pipeline runs mechanically -- Phase 2 covers actual training/eval).

**Before running**: Settings > Accelerator > GPU T4x2 (or P100). Settings > Internet > On. Settings > Persistence > "Variables and files". Settings > Environment > "Pin to original environment".

**Only 3 code cells, deliberately.** Earlier versions of this notebook had ~12 small cells, and repeatedly ran into a real workflow problem: pasting an updated cell as a *new* cell (rather than replacing the old one) left a stale duplicate further down that kept silently executing instead of the fix. Consolidating everything into 3 cells makes that class of mistake much harder to make -- if something needs updating, there's only one place per stage to look.

**Every cell is safe to re-run, including after a session restart or Run All.** Kaggle's "pin to original environment" reverts to the base image's default packages on every fresh session -- anything pip-installed outside `/kaggle/working` does NOT survive a restart, only files under `/kaggle/working` do -- so Cell 1 always reinstalls (fast, since pip no-ops on already-satisfied packages).

**Disk space**: Kaggle's `/kaggle/working` quota (~20GB) can't hold the raw downloaded DENTEX dataset (11.8GB) plus a full extraction of the largest archive (`training_data.zip`, 10.9GB compressed) at the same time -- confirmed on a real run ("OSError: No space left on device", including for something as small as creating an empty directory once the disk filled up completely). Cell 2 never extracts `training_data.zip`/`test_data.zip` (reads their stats directly via `zipfile`, no extraction needed) and deletes both immediately after, freeing ~11.6GB. This means a future session will need to re-download those two archives again even with file persistence on -- unavoidable given the disk quota, not a bug.

Everything here -- the Pillow/legacy-PIL-constant issue, the PIL._util.is_directory issue, the pycocotools `_mask` extension, the exact dataset paths, the Swin-B checkpoint size mismatch, the disk-space handling -- has been verified end-to-end against a local Python 3.9/3.12 environment and real downloaded data before ever being run on Kaggle. If something still fails here, it's a genuinely new issue, not a repeat of one already chased down; paste the full traceback rather than just the last few lines, since some earlier bugs here only became clear from output further up (shape-mismatch and disk-space warnings in particular are easy to miss in a long log).

## Cell 1 -- environment setup

GPU check, clone/update the repo, install dependencies, fix the bundled `pycocotools` extension, verify the full import chain.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in notebook Settings before proceeding.'

# Clone (or update) the repo. Idempotent: safe to re-run after a restart --
# pulls latest instead of re-cloning if /kaggle/working/repo already exists.
REPO_URL = 'https://github.com/christopherh-88/HierarchicalDet.git'
%cd /kaggle/working
import os
if os.path.isdir('repo/.git'):
    %cd repo
    !git pull
else:
    !git clone $REPO_URL repo
    %cd repo

# Install dependencies. Always re-run this on a fresh session -- pip installs
# outside /kaggle/working don't persist across restarts here.
#
# Pillow note: this old detectron2 snapshot uses PIL.Image.LINEAR (removed in
# Pillow 10) and relies on PIL._util.is_directory (also removed in newer
# Pillow, needed transitively via torchvision -> PIL.ImageFont). Both are now
# fixed at the source in the repo itself (detectron2/utils/env.py and
# several other files, redundantly) rather than via a pip pin -- pinning
# pillow<10 was tried and never reliably worked on Kaggle across multiple
# attempts (wrong interpreter for `!pip install`, then PEP-668 blocking
# installs silently, then a reinstall reporting success while doing
# nothing). Nothing needs to happen here for Pillow anymore.
import sys, subprocess
print('sys.executable:', sys.executable)

def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + args
    try:
        subprocess.run(cmd + ['--break-system-packages'], check=True,
                        capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        if 'no such option' in (e.stderr or '').lower() or 'unrecognized arguments' in (e.stderr or '').lower():
            # older pip without PEP 668 support -- retry without the flag
            subprocess.run(cmd, check=True, capture_output=True, text=True)
        else:
            print(e.stdout)
            print(e.stderr)
            raise

# Exclude torch/torchvision/torchaudio -- Kaggle ships a CUDA-matched build
# already, don't clobber it.
subprocess.run(
    "grep -v -i -E '^(torch|torchvision|torchaudio)$' requirements.txt > /tmp/reqs_no_torch.txt",
    shell=True, check=True,
)
pip_install(['-r', '/tmp/reqs_no_torch.txt'])
pip_install(['huggingface_hub'])
print('Dependencies installed.')

# Bundled pycocotools/ (custom categories_1/2/3 3-tier format) ships without
# its compiled _mask Cython extension, and repo-root imports always shadow
# the pip-installed pycocotools package. Copy the compiled extension for this
# interpreter/platform from the pip install into the bundled package.
import site, glob, shutil
site_pkgs = site.getsitepackages()[0]
mask_so = glob.glob(f'{site_pkgs}/pycocotools/_mask*.so')
assert mask_so, f'no compiled _mask extension found under {site_pkgs}/pycocotools'
shutil.copy(mask_so[0], 'pycocotools/')
print('copied', mask_so[0])

# Verify the full import chain, same checks as local setup.sh
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from hierarchialdet import add_diffusiondet_config, DiffusionDetWithTTA
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper
from pycocotools.coco import COCO
import evaluator
print('All imports OK.')

## Cell 2 -- download DENTEX, verify all three annotation tiers, extract + normalize training data

Verified 2026-07-14 by directly inspecting the real dataset (not guessed --
`training_data.zip` is 10.9GB, so its internal file listing was pulled via an
HTTP range request against just the zip's central directory, and the three
annotation JSONs were fetched via their own byte ranges, without downloading
the whole archive):

- `training_data.zip` contains **four** subfolders, not three:
  `quadrant/` (693 images, `train_quadrant.json`, flat `category_id`, 4 quadrant classes),
  `quadrant_enumeration/` (634 images, `train_quadrant_enumeration.json`, `category_id_1/2`, 4+8 classes),
  `quadrant-enumeration-disease/` (705 images, `train_quadrant_enumeration_disease.json`, `category_id_1/2/3`, 4+8+4 classes -- diagnosis classes are Impacted/Caries/Periapical Lesion/Deep Caries, matching the paper), and
  `unlabelled/` (1571 images, no annotations at all -- not documented in either README. Confirmed via the Phase 1 dataset audit that this is exactly the data the HierarchicalDet paper uses for SimMIM self-supervised pretraining of the Swin backbone before the main hierarchical training -- see docs/phase1_dataset_audit.md -- but the current `nonpretrain.yaml` config doesn't use it, using plain ImageNet-22k Swin-B weights instead).
- `validation_data.zip` contains only images, at `validation_data/quadrant_enumeration_disease/xrays/val_N.png` (50 of them). The matching annotations are **not** in the zip -- they're the separate top-level `DENTEX/validation_triple.json` (50 images, 182 annotations, confirmed to reference these exact filenames).
- **`test_data.zip` uses a completely different, incompatible format**: per-image LabelMe polygon JSON files (`disease/label/test_N.json`) with Turkish-language diagnosis strings baked into a single `label` field (e.g. `"1-çürük-15"` = quadrant-diagnosis-tooth), images at `disease/input/test_N.png`. Not loadable by `register_coco_instances()` or `DiffusionDetDatasetMapper` as-is -- using the test split requires a LabelMe-to-COCO conversion script that doesn't exist yet anywhere in this codebase (Phase 2 scope; this cell only reports file counts).
- Category ID *numbering* is not consistent across files -- always read category names from each file's own `categories_N` list rather than assuming a fixed id-to-name mapping.
- **The official dataloader (`register_coco_instances` + the bundled `pycocotools`) cannot load the raw `quadrant` or `quadrant_enumeration` training tiers as-is** -- confirmed in the Phase 1 audit: both `detectron2/data/datasets/coco.py:76` and `pycocotools/coco.py:184` unconditionally assume all three `categories_1/2/3` keys exist, which only the fully-annotated tier's raw JSON has. **Fixed in Phase 2** (not by patching pycocotools/detectron2, but by normalizing the raw JSONs into the same schema -- see `docs/phase2_dataloader_fix.md`): this cell now extracts all three training tiers and runs `tools/normalize_dentex_tiers.py`, producing `QUADRANT_TRAIN_JSON`/`QUADRANT_ENUM_TRAIN_JSON` that load through the unmodified official pipeline.
- **38-40% of test/validation images are byte-identical to a training-tier image under a different filename** (confirmed via MD5 hash, not just filename coincidence). Checked against both the HierarchicalDet paper and the companion DENTEX dataset paper directly: Tiers 1-2 have no official train/test split at all (pooled entirely into "training and development"), and neither paper addresses cross-tier overlap. This is very likely present in the original authors' own results too, not a reproduction error -- see docs/phase1_dataset_audit.md.

`training_data.zip` and `test_data.zip` are read directly (via `zipfile`) and then **deleted** after extracting only the training-tier subfolders actually needed -- see the disk-space note in the top markdown cell. `validation_data.zip` (small, 149.5MB) is extracted to disk in full, since that's where the smoke-test image comes from.

In [ ]:
import os, json, shutil, zipfile
from huggingface_hub import snapshot_download

RAW_DIR = '/kaggle/working/dentex_raw'
EXTRACT_DIR = '/kaggle/working/dentex_extracted'
os.makedirs(EXTRACT_DIR, exist_ok=True)

dataset_dir = snapshot_download(
    repo_id='ibrahimhamamci/DENTEX',
    repo_type='dataset',
    local_dir=RAW_DIR,
)
print('downloaded to', dataset_dir)

TRAINING_ZIP = os.path.join(dataset_dir, 'DENTEX/training_data.zip')
VALIDATION_ZIP = os.path.join(dataset_dir, 'DENTEX/validation_data.zip')
TEST_ZIP = os.path.join(dataset_dir, 'DENTEX/test_data.zip')

# Clean up any partial extraction left over from an earlier run that hit the
# disk-space error, to reclaim space before anything else runs.
partial_training_dir = os.path.join(EXTRACT_DIR, 'training_data')
if os.path.isdir(partial_training_dir):
    print('removing partial training_data extraction from a previous run to reclaim disk space...')
    shutil.rmtree(partial_training_dir)

# Only validation_data.zip gets fully extracted -- small, and it's where the
# smoke-test image comes from.
marker = os.path.join(EXTRACT_DIR, '.DENTEX_validation_data.zip.extracted')
if os.path.exists(marker):
    print('already extracted: validation_data.zip')
elif not os.path.exists(VALIDATION_ZIP):
    print('MISSING (not downloaded?):', VALIDATION_ZIP)
else:
    print('extracting', VALIDATION_ZIP, '...')
    with zipfile.ZipFile(VALIDATION_ZIP) as zf:
        zf.extractall(EXTRACT_DIR)
    open(marker, 'w').close()
    print('done: validation_data.zip')

VALIDATION_TRIPLE_PATH = os.path.join(dataset_dir, 'DENTEX', 'validation_triple.json')
VALIDATION_IMAGE_DIR = os.path.join(EXTRACT_DIR, 'validation_data', 'quadrant_enumeration_disease', 'xrays')

TRAINING_TIER_MEMBERS = {
    'quadrant (train)': ('training_data/quadrant/train_quadrant.json', 'training_data/quadrant/xrays/'),
    'quadrant-enumeration (train)': ('training_data/quadrant_enumeration/train_quadrant_enumeration.json', 'training_data/quadrant_enumeration/xrays/'),
    'quadrant-enumeration-diagnosis (train)': ('training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json', 'training_data/quadrant-enumeration-disease/xrays/'),
}
UNLABELLED_PREFIX = 'training_data/unlabelled/xrays/'

any_missing = False

if os.path.exists(TRAINING_ZIP):
    with zipfile.ZipFile(TRAINING_ZIP) as zf:
        names = zf.namelist()
        for tier, (json_member, img_prefix) in TRAINING_TIER_MEMBERS.items():
            if json_member not in names:
                print(tier, '-> MISSING member in zip:', json_member)
                any_missing = True
                continue
            d = json.loads(zf.read(json_member))
            n_images_in_zip = sum(1 for n in names if n.startswith(img_prefix) and n.endswith('.png'))
            print(tier, '->', json_member, '(inside training_data.zip, not extracted)')
            print('  images (JSON):', len(d.get('images', [])), ' images (zip entries):', n_images_in_zip, ' annotations:', len(d.get('annotations', [])))
            for k in d:
                if k.startswith('categor'):
                    print('  ', k, ':', [c['name'] for c in d[k]])
        n_unlabelled = sum(1 for n in names if n.startswith(UNLABELLED_PREFIX) and n.endswith('.png'))
        print('unlabelled xrays in zip (no annotations, not used by default pipeline):', n_unlabelled)

        # Extract all three training tiers (JSON + images, ~2.4-2.7GB each)
        # BEFORE training_data.zip gets deleted for disk space. All three are
        # now usable: quadrant-enumeration-disease loads through the official
        # dataloader as-is, and quadrant/quadrant_enumeration are made
        # loadable via tools/normalize_dentex_tiers.py below (see
        # docs/phase2_dataloader_fix.md -- the official pycocotools/
        # detectron2 loader genuinely can't parse their raw schema, so they're
        # normalized into the same categories_1/2/3 + category_id_1/2/3 shape
        # quadrant-enumeration-disease already uses, not patched in place).
        # Each extraction is idempotent (marker-file gated) so re-running
        # this cell after a restart doesn't re-extract what's already there.
        for tier_dir, marker_name in [
            ('training_data/quadrant/', '.DENTEX_training_data_quadrant_tier.extracted'),
            ('training_data/quadrant_enumeration/', '.DENTEX_training_data_quadrant_enumeration_tier.extracted'),
            ('training_data/quadrant-enumeration-disease/', '.DENTEX_training_data_disease_tier.extracted'),
        ]:
            tier_marker = os.path.join(EXTRACT_DIR, marker_name)
            if os.path.exists(tier_marker):
                print('already extracted:', tier_dir)
            else:
                tier_members = [n for n in names if n.startswith(tier_dir)]
                print(f'extracting {len(tier_members)} members from {tier_dir}...')
                zf.extractall(EXTRACT_DIR, members=tier_members)
                open(tier_marker, 'w').close()
                print('done extracting', tier_dir)
else:
    print('MISSING:', TRAINING_ZIP)
    any_missing = True

if os.path.exists(VALIDATION_TRIPLE_PATH):
    d = json.load(open(VALIDATION_TRIPLE_PATH))
    n_extracted = len([f for f in os.listdir(VALIDATION_IMAGE_DIR) if f.endswith('.png')]) if os.path.isdir(VALIDATION_IMAGE_DIR) else 0
    print('quadrant-enumeration-diagnosis (validation) ->', VALIDATION_TRIPLE_PATH)
    print('  images (JSON):', len(d.get('images', [])), ' images (extracted to disk):', n_extracted, ' annotations:', len(d.get('annotations', [])))
    for k in d:
        if k.startswith('categor'):
            print('  ', k, ':', [c['name'] for c in d[k]])
else:
    print('MISSING:', VALIDATION_TRIPLE_PATH)
    any_missing = True

print()
print('=== TEST SET: LabelMe polygons, converted to COCO below ===')
# The test split ships as per-image LabelMe JSONs with Turkish diagnosis
# strings rather than COCO, so it cannot be registered or evaluated as-is.
# tools/labelme_to_coco.py converts it (Cell 5). Extract the split here,
# while the zip still exists -- it is deleted a few lines below to stay
# inside Kaggle's disk quota, and re-downloading it costs another session.
TEST_LABEL_DIR = os.path.join(EXTRACT_DIR, 'test_data', 'disease', 'label')
TEST_IMAGE_DIR = os.path.join(EXTRACT_DIR, 'test_data', 'disease', 'input')
if os.path.exists(TEST_ZIP):
    with zipfile.ZipFile(TEST_ZIP) as zf:
        names = zf.namelist()
        n_test_images = sum(1 for n in names if n.startswith('disease/input/') and n.endswith('.png'))
        n_test_labels = sum(1 for n in names if n.startswith('disease/label/') and n.endswith('.json'))
        print(f'{n_test_images} test images, {n_test_labels} per-image LabelMe label files (Turkish diagnosis strings)')
        test_marker = os.path.join(EXTRACT_DIR, '.DENTEX_test_data.extracted')
        if os.path.exists(test_marker):
            print('already extracted: test_data')
        else:
            test_members = [n for n in names if n.startswith('disease/')]
            print(f'extracting {len(test_members)} test members...')
            zf.extractall(os.path.join(EXTRACT_DIR, 'test_data'), members=test_members)
            open(test_marker, 'w').close()
            print('done extracting test_data')
elif os.path.isdir(TEST_LABEL_DIR):
    print('test_data.zip already deleted; using the earlier extraction at', TEST_LABEL_DIR)
else:
    print('MISSING:', TEST_ZIP)

if any_missing:
    print()
    print('Some expected paths/members were missing -- inspect the zip contents directly, e.g.:')
    print('  zipfile.ZipFile(TRAINING_ZIP).namelist()[:50]')

# Free disk space: we've now read/extracted everything needed out of
# TRAINING_ZIP and TEST_ZIP above and don't need the zip files themselves
# anymore. Kaggle's /kaggle/working quota is small enough that leaving all
# three raw zips (11.8GB combined) on disk starves later steps (confirmed on
# a real run -- even os.makedirs() for a new empty directory failed with
# "No space left on device" once the disk filled up completely).
for big_zip in (TRAINING_ZIP, TEST_ZIP):
    if os.path.exists(big_zip):
        size_gb = os.path.getsize(big_zip) / 1e9
        os.remove(big_zip)
        print(f'freed {size_gb:.1f}GB by removing {big_zip} (already read everything needed from it above)')

# Normalize quadrant/quadrant_enumeration into the same 3-tier schema
# quadrant-enumeration-disease already uses (categories_1/2/3,
# category_id_1/2/3 with null for tiers that don't apply), so all three
# tiers can be loaded through the same register_coco_instances() path for
# the hierarchical training curriculum. See docs/phase2_dataloader_fix.md
# for why this is a data-normalization fix rather than a pycocotools/
# detectron2 patch, and the id-remap bug it specifically catches (quadrant's
# own category ids don't match the other tiers' for the same 4 classes --
# verified against real local data, 0 mismatches across 2,772 + 18,095
# annotations, before this script was ever written into this notebook).
import subprocess
QUADRANT_RAW_JSON = os.path.join(EXTRACT_DIR, 'training_data', 'quadrant', 'train_quadrant.json')
QUADRANT_IMG_DIR = os.path.join(EXTRACT_DIR, 'training_data', 'quadrant', 'xrays')
QUADRANT_ENUM_RAW_JSON = os.path.join(EXTRACT_DIR, 'training_data', 'quadrant_enumeration', 'train_quadrant_enumeration.json')
QUADRANT_ENUM_IMG_DIR = os.path.join(EXTRACT_DIR, 'training_data', 'quadrant_enumeration', 'xrays')
DISEASE_JSON = os.path.join(EXTRACT_DIR, 'training_data', 'quadrant-enumeration-disease', 'train_quadrant_enumeration_disease.json')
DISEASE_IMG_DIR = os.path.join(EXTRACT_DIR, 'training_data', 'quadrant-enumeration-disease', 'xrays')
NORMALIZED_DIR = os.path.join(EXTRACT_DIR, 'training_data', 'normalized')

norm_result = subprocess.run([
    sys.executable, 'tools/normalize_dentex_tiers.py',
    '--full-tier-json', DISEASE_JSON,
    '--quadrant-json', QUADRANT_RAW_JSON,
    '--quadrant-enumeration-json', QUADRANT_ENUM_RAW_JSON,
    '--out-dir', NORMALIZED_DIR,
], capture_output=True, text=True)
print(norm_result.stdout)
if norm_result.returncode != 0:
    print(norm_result.stderr)
assert norm_result.returncode == 0, 'tools/normalize_dentex_tiers.py failed -- see stderr above.'

QUADRANT_TRAIN_JSON = os.path.join(NORMALIZED_DIR, 'train_quadrant_normalized.json')
QUADRANT_ENUM_TRAIN_JSON = os.path.join(NORMALIZED_DIR, 'train_quadrant_enumeration_normalized.json')

# TRAIN_JSON/TRAIN_IMG_DIR (unchanged names, used by Cell 3's smoke test and
# Cell 4's benchmark below) still point at quadrant-enumeration-disease --
# the fully-annotated tier. Use QUADRANT_TRAIN_JSON/QUADRANT_IMG_DIR and
# QUADRANT_ENUM_TRAIN_JSON/QUADRANT_ENUM_IMG_DIR for tier-0/tier-1 of the
# hierarchical curriculum.
TRAIN_JSON = DISEASE_JSON
TRAIN_IMG_DIR = DISEASE_IMG_DIR
!df -h /kaggle/working

## Cell 3 -- single-image forward-pass smoke test

No pretrained HierarchicalDet weights exist publicly, so this uses the public
Swin-B ImageNet-22k backbone per `configs/diffdet.custom.swinbase.nonpretrain.yaml`,
with the detection head randomly initialized. The goal is only to prove the
model builds and runs a forward pass without crashing -- not to produce
meaningful detections (expect "detected 0 instances").

Verified locally end-to-end on 2026-07-14 (Python 3.12, CPU, a real
downloaded validation image) before this was ever run on Kaggle. This caught
a real bug, now fixed: `configs/diffdet.custom.swinbase.nonpretrain.yaml` had
`SWIN.SIZE: L-22k` (Large, embed_dim=192) while its own `MODEL.WEIGHTS`
pointed at a Base-sized checkpoint filename (embed_dim=128) -- an internal
inconsistency in the original authors' own config (the file is literally
named "swinbase"). This caused 325 silent shape-mismatch warnings and meant
the backbone was effectively randomly initialized -- easy to miss since
detectron2 logs shape mismatches as warnings, not errors. Fixed by
correcting `SIZE` to `B-22k`; confirmed zero shape mismatches after, with
only the expected non-backbone components (diffusion-process buffers,
detection head) reported as unmatched.

In [ ]:
# Download the Swin-B backbone (skipped if already present).
# IMPORTANT: keep the .pth extension -- detectron2's DetectionCheckpointer
# dispatches purely on file extension: ".pkl" is parsed as a Caffe2-style
# pickle blob, which would silently mis-load a native torch.save() file.
os.makedirs('models', exist_ok=True)
WEIGHTS_PATH = 'models/swin_base_patch4_window7_224_22k.pth'
if not os.path.exists(WEIGHTS_PATH):
    !wget -q -O $WEIGHTS_PATH https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_base_patch4_window7_224_22k.pth
# !wget doesn't raise on failure (same silent-failure class of bug as !pip
# install earlier in this notebook's history) -- verify the file actually
# landed and isn't suspiciously small (e.g. an HTML error page saved as
# bytes) before trusting it.
assert os.path.exists(WEIGHTS_PATH), f'{WEIGHTS_PATH} was not created -- the wget download above failed silently.'
assert os.path.getsize(WEIGHTS_PATH) > 1_000_000, f'{WEIGHTS_PATH} is suspiciously small ({os.path.getsize(WEIGHTS_PATH)} bytes) -- likely a failed/partial download.'
print(WEIGHTS_PATH, os.path.getsize(WEIGHTS_PATH), 'bytes')

# Use a known-real validation image and the real category names
# (DENTEX_CATEGORY_JSON -> validation_triple.json) instead of a placeholder.
sample_image = os.path.join(VALIDATION_IMAGE_DIR, 'val_0.png')
assert os.path.exists(sample_image), f'{sample_image} not found -- check Cell 2 ran successfully.'
print('using sample image:', sample_image)

import subprocess
env = dict(os.environ, DENTEX_CATEGORY_JSON=VALIDATION_TRIPLE_PATH)
result = subprocess.run([
    sys.executable, 'demo.py',
    '--config-file', 'configs/diffdet.custom.swinbase.nonpretrain.yaml',
    '--input', sample_image,
    '--output', '/kaggle/working/smoke_test_output.jpg',
    '--opts', 'MODEL.WEIGHTS', WEIGHTS_PATH, 'MODEL.DEVICE', 'cuda',
], env=env)
assert result.returncode == 0, f'demo.py exited with code {result.returncode}'

# demo.py now also writes a raw score file (boxes/scores/per-tier classes)
# alongside the visualization -- the roadmap's Phase 0 deliverable asks for
# "terminal output, detection visualizations, AND raw score files", and the
# original demo.py discarded the raw predictions dict entirely, only ever
# saving the rendered image. Confirm it's actually there.
scores_path = '/kaggle/working/smoke_test_output_scores.json'
assert os.path.exists(scores_path), f'{scores_path} was not created -- demo.py raw-score patch may be missing.'
print('Smoke test complete:')
print(' visualization:', '/kaggle/working/smoke_test_output.jpg')
print(' raw scores:   ', scores_path)
print(open(scores_path).read())

## Cell 4 -- real training-loop throughput benchmark

The roadmap's Phase 0 asks to "estimate GPU compute cost before proceeding"
given no pretrained HierarchicalDet weights exist. Earlier this was a
generic, unsourced "~12-20h on A100/V100" guess -- not validated against the
config actually being used (`SOLVER.MAX_ITER=40000`, `SOLVER.IMS_PER_BATCH=2`)
or against real Kaggle hardware. This cell replaces that guess with a real
measurement: runs the *actual* training code path (`train_net_patched.py`'s
`Trainer`, same model/optimizer/mapper Phase 2 will use, not a simplified
stand-in) for 30 real iterations and reads detectron2's own post-warmup
timing summary, then extrapolates to the real iteration count.

`train_net_patched.py` is a patched copy of `train_net.py`, needed because
the original hardcodes two things that don't work here: (1) it uses the
*unpatched* `hierarchialdet.DiffusionDetDatasetMapper`, which crashes on
construction (hardcoded non-public noisy-box paths) -- fixed by using
`hierarchialdet.dataset_mapper_patched.DiffusionDetDatasetMapper` instead;
(2) it hardcodes dataset registration to non-public paths -- fixed by
reading `TRAIN_JSON`/`TRAIN_IMG_DIR`/`VAL_JSON`/`VAL_IMG_DIR` from
environment variables.

Only the `quadrant-enumeration-disease` tier is benchmarked below (all three
training tiers are now loadable as of Cell 2's normalization step -- see
`docs/phase2_dataloader_fix.md` -- but only the diagnosis tier has been
throughput-tested so far). The 3x extrapolation to the full hierarchical
sequence below is therefore approximate, not a real measurement of all three
tiers; the actual 3-stage curriculum training run (using `TRAIN_JSON`,
`QUADRANT_TRAIN_JSON`, `QUADRANT_ENUM_TRAIN_JSON` from Cell 2 in sequence) is
still to be done.

Verified locally beforehand (Python 3.12, CPU, real `quadrant-enumeration-disease`
data) that the entire path -- dataset registration via env vars, the patched
mapper, model/optimizer construction, and a real backward pass with real
loss values -- runs end to end without error, and that the log-parsing regex
below correctly extracts detectron2's own timing summary line. This also
surfaced one real missing dependency (`tensorboard`, needed by detectron2's
default training writers, never hit before since only inference had been
run) -- added to `requirements.txt`.

In [ ]:
import re

BENCH_ITERS = 30  # comfortably past IterationTimer's 3-iteration warmup exclusion
BENCH_OUTPUT_DIR = '/kaggle/working/train_bench_output'
os.makedirs(BENCH_OUTPUT_DIR, exist_ok=True)

env = dict(
    os.environ,
    TRAIN_JSON=TRAIN_JSON,
    TRAIN_IMG_DIR=TRAIN_IMG_DIR,
    VAL_JSON=VALIDATION_TRIPLE_PATH,
    VAL_IMG_DIR=VALIDATION_IMAGE_DIR,
)

def run_benchmark(ims_per_batch):
    cmd = [
        sys.executable, 'train_net_patched.py',
        '--config-file', 'configs/diffdet.custom.swinbase.nonpretrain.yaml',
        '--num-gpus', '1',
        'MODEL.WEIGHTS', WEIGHTS_PATH,
        'MODEL.DEVICE', 'cuda',
        'SOLVER.MAX_ITER', str(BENCH_ITERS),
        'SOLVER.IMS_PER_BATCH', str(ims_per_batch),
        'OUTPUT_DIR', BENCH_OUTPUT_DIR,
    ]
    print('running:', ' '.join(cmd))
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    return result, result.stdout + result.stderr

# configs/Base-DiffusionDet.yaml's default SOLVER.IMS_PER_BATCH is 2 (not
# overridden by nonpretrain.yaml). Try that first since it's the real
# config's real setting -- but Swin-B + FPN + 1000 diffusion proposals at
# batch 2 can exceed a single T4's 14.56GB VRAM (confirmed via a real
# CUDA OutOfMemoryError on an actual run), so fall back to batch 1 if it
# does. Either way this is a real, worth-reporting finding for Phase 2, not
# just a benchmark inconvenience -- it means training may need batch 1 (or
# gradient checkpointing / a bigger GPU) to run at all on this hardware.
REAL_IMS_PER_BATCH_CONFIG = 2  # what the config actually specifies
result, log = run_benchmark(REAL_IMS_PER_BATCH_CONFIG)
ims_per_batch_used = REAL_IMS_PER_BATCH_CONFIG
oom = result.returncode != 0 and 'CUDA out of memory' in log
if oom:
    print()
    print(f'*** SOLVER.IMS_PER_BATCH={REAL_IMS_PER_BATCH_CONFIG} (the config default) hit CUDA OOM on this GPU. ***')
    print('Retrying at SOLVER.IMS_PER_BATCH=1 -- Phase 2 training will need this same adjustment')
    print('(or gradient checkpointing / a larger GPU) to actually run on this hardware.')
    result, log = run_benchmark(1)
    ims_per_batch_used = 1

print(log[-4000:])  # tail only -- the full config dump is huge
assert result.returncode == 0, f'train_net_patched.py exited with code {result.returncode} even at batch size {ims_per_batch_used}'

match = re.search(r'Overall training speed: (\d+) iterations in [\d:]+ \(([\d.]+) s / it\)', log)
assert match, 'Could not find the "Overall training speed" summary line -- check the printed tail above for what actually happened.'
n_iters_measured = int(match.group(1))
s_per_iter = float(match.group(2))

REAL_MAX_ITER = 40000  # configs/diffdet.custom.swinbase.nonpretrain.yaml SOLVER.MAX_ITER
single_tier_hours = s_per_iter * REAL_MAX_ITER / 3600

print()
print(f'Measured: {s_per_iter:.4f} s/iter over {n_iters_measured} post-warmup iterations,')
print(f'  batch size {ims_per_batch_used} (config default is {REAL_IMS_PER_BATCH_CONFIG}{" -- reduced due to CUDA OOM, see above" if oom else ""}), single GPU, quadrant-enumeration-disease tier data.')
print(f'Extrapolated to the real config (SOLVER.MAX_ITER={REAL_MAX_ITER}): {single_tier_hours:.2f} hours for one tier, single GPU.')
print(f'Full 3-tier hierarchical sequence (tier-0 + tier-1 + tier-2, run separately): roughly {3*single_tier_hours:.2f} hours --')
print('  approximate, since only the diagnosis tier was actually measured (tier-0/tier-1 need the')
print('  dataloader patch documented in docs/phase1_dataset_audit.md before they can run at all).')
if oom:
    print(f'Note: this measures batch size {ims_per_batch_used}, not the config\'s stated {REAL_IMS_PER_BATCH_CONFIG} --')
    print('  a real memory constraint, not just a benchmark adjustment. Phase 2 training at the full')
    print('  config needs either gradient checkpointing (MODEL.SWIN.USE_CHECKPOINT, untested) or a larger GPU.')
print('Not tested: 2-GPU scaling (Kaggle offers T4x2) -- would likely reduce wall-clock somewhat but not exactly halve it due to DDP overhead.')

## Cell 5 -- convert the test split to COCO

The 250-image test split is the only one DENTEX does not ship as COCO: it is
per-image LabelMe polygons whose single `label` field packs quadrant, a Turkish
diagnosis string and the FDI tooth number together (`"1-curuk-15"`).
`tools/labelme_to_coco.py` converts it into the same normalized 3-tier schema
everything else uses, taking the category ids from the training tier so class
indices match what the model was trained with.

It runs in report mode first. If any label string is not covered by the
script's alias table the run stops and prints the offending strings rather than
guessing -- extend `DIAGNOSIS_ALIASES` in the script and re-run. A silently
mislabelled test set would corrupt every number downstream.

In [ ]:
TEST_COCO_JSON = os.path.join(EXTRACT_DIR, 'test_data', 'test_quadrant_enumeration_disease_coco.json')

# Report mode: prints every distinct raw label and how it parsed. Read this
# output before trusting the converted file.
report = subprocess.run([
    sys.executable, 'tools/labelme_to_coco.py',
    '--label-dir', TEST_LABEL_DIR,
    '--image-dir', TEST_IMAGE_DIR,
    '--canonical-json', DISEASE_JSON,
    '--report-only',
], capture_output=True, text=True)
print(report.stdout[-6000:])
print(report.stderr[-2000:])
assert report.returncode == 0, 'label parsing failed -- see the report above'

convert = subprocess.run([
    sys.executable, 'tools/labelme_to_coco.py',
    '--label-dir', TEST_LABEL_DIR,
    '--image-dir', TEST_IMAGE_DIR,
    '--canonical-json', DISEASE_JSON,
    '--out-json', TEST_COCO_JSON,
], capture_output=True, text=True)
print(convert.stdout[-3000:])
print(convert.stderr[-2000:])
assert convert.returncode == 0, 'conversion failed'

# Confirm the converted file loads through the *official* dataloader, the same
# way the training tiers do -- a COCO file that parses as JSON but not as a
# detectron2 dataset would only fail much later, mid-evaluation.
import importlib
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
if 'dentex_test' in DatasetCatalog.list():
    DatasetCatalog.remove('dentex_test'); MetadataCatalog.remove('dentex_test')
register_coco_instances('dentex_test', {}, TEST_COCO_JSON, TEST_IMAGE_DIR)
test_dicts = DatasetCatalog.get('dentex_test')
meta = MetadataCatalog.get('dentex_test')
print('loaded', len(test_dicts), 'test images through register_coco_instances')
print('thing_classes1/2/3:', meta.thing_classes1, meta.thing_classes2, meta.thing_classes3)

## Cell 6 -- the 3-stage hierarchical curriculum (Phase 3, the long run)

`tools/run_curriculum.py` runs the whole thing: train on quadrant, run that
model over the enumeration tier's images to produce noisy boxes, train on
enumeration seeded with them, repeat into diagnosis. Weight transfer is the
previous stage's `model_final.pth`; the noisy boxes reach the model through
`NOISY_BOX_TRAIN` / `NOISY_BOX_VAL`, which the patched dataset mapper reads.

**This does not fit in one Kaggle session.** At the Phase 0 measured throughput
each stage at the config's 40k iterations is roughly 10 hours, so ~31 hours
total against a 12h-per-session, 30h-per-week GPU quota. Every stage is skipped
if its `model_final.pth` already exists and training itself resumes from the
last checkpoint, so the intended pattern is: run this cell, let the session
expire, and run it again next session -- as long as `--output-root` is on
persistent storage (a Kaggle Dataset, or `/kaggle/working` with persistence on).

Set `MAX_ITER` below to a small number first to prove the whole chain runs end
to end (both dumps, all three stages) before committing days of GPU quota to
it. `--dry-run` prints the exact command sequence without running anything.

`IMS_PER_BATCH=1` is not a preference: batch 2, the config default, hit a real
CUDA OOM on a T4 in the Phase 0 benchmark.

In [ ]:
CURRICULUM_ROOT = '/kaggle/working/curriculum'
MAX_ITER = 200      # <-- smoke value. Set to 40000 for the real run.
EVAL_PERIOD = 100   # <-- and to 2000+ for the real run.

curriculum_cmd = [
    sys.executable, 'tools/run_curriculum.py',
    '--data-root', os.path.join(EXTRACT_DIR, 'training_data'),
    '--normalized-dir', NORMALIZED_DIR,
    '--val-json', VALIDATION_TRIPLE_PATH,
    '--val-images', VALIDATION_IMAGE_DIR,
    '--output-root', CURRICULUM_ROOT,
    '--init-weights', WEIGHTS_PATH,
    '--config-file', 'configs/diffdet.custom.swinbase.nonpretrain.yaml',
    '--ims-per-batch', '1',
    '--max-iter', str(MAX_ITER),
    '--eval-period', str(EVAL_PERIOD),
]

print('planned commands:')
subprocess.run(curriculum_cmd + ['--dry-run'], check=True)

# Streams the full log; the run continues across cell restarts via the
# per-stage skip logic described above.
subprocess.run(curriculum_cmd, check=True)

FINAL_WEIGHTS = os.path.join(CURRICULUM_ROOT, 'stage2_diagnosis', 'model_final.pth')
print('final checkpoint:', FINAL_WEIGHTS, os.path.exists(FINAL_WEIGHTS))

## Cell 7 -- per-tier evaluation, cross-check, error analysis, visualizations

Four things, in order:

1. `tools/evaluate_tiers.py` scores the final checkpoint with the **authors'
   own** evaluator, reporting AP / AP50 / AP75 and per-class AP for all three
   tiers in one pass, on validation and on the converted test split.
2. `tools/dump_predictions.py` writes raw predictions plus per-image runtime and
   failure counts (no detections, degenerate boxes, boxes out of frame).
3. `tools/coco_eval_standalone.py` re-scores the same predictions with stock
   `pycocotools`, as a check on the repo's forked evaluator. The two columns
   should agree closely; if they do not, that disagreement is itself a finding.
4. `tools/error_analysis.py` and `tools/visualize_predictions.py` produce the
   failure taxonomy (missed teeth, false positives, wrong diagnosis on a
   correctly found tooth, boxes spanning multiple teeth) and the GT-vs-prediction
   overlays.

In [ ]:
EVAL_ROOT = '/kaggle/working/eval'
os.makedirs(EVAL_ROOT, exist_ok=True)

SPLITS = {
    'validation': (VALIDATION_TRIPLE_PATH, VALIDATION_IMAGE_DIR),
    'test': (TEST_COCO_JSON, TEST_IMAGE_DIR),
}

for split_name, (split_json, split_images) in SPLITS.items():
    out_dir = os.path.join(EVAL_ROOT, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 1. official evaluator, all three tiers
    subprocess.run([
        sys.executable, 'tools/evaluate_tiers.py',
        '--weights', FINAL_WEIGHTS,
        '--json', split_json,
        '--image-dir', split_images,
        '--tier', '2',
        '--output-dir', out_dir,
        '--label', 'HierarchicalDet ({})'.format(split_name),
    ], env=dict(os.environ, TIER='2'), check=True)

    # 2. raw predictions + runtime + failure counts
    predictions = os.path.join(out_dir, 'predictions.json')
    subprocess.run([
        sys.executable, 'tools/dump_predictions.py',
        '--weights', FINAL_WEIGHTS,
        '--json', split_json,
        '--image-dir', split_images,
        '--tier', '2',
        '--output', predictions,
    ], env=dict(os.environ, TIER='2'), check=True)

    # 3. independent COCOeval cross-check
    subprocess.run([
        sys.executable, 'tools/coco_eval_standalone.py',
        '--gt-json', split_json,
        '--predictions', predictions,
        '--tier', '2', '--all-tiers',
        '--output', os.path.join(out_dir, 'standalone_metrics.json'),
    ], check=True)

    # 4. error taxonomy + qualitative overlays
    subprocess.run([
        sys.executable, 'tools/error_analysis.py',
        '--gt-json', split_json,
        '--predictions', predictions,
        '--tier', '2',
        '--output', os.path.join(out_dir, 'error_analysis.json'),
    ], check=True)
    subprocess.run([
        sys.executable, 'tools/visualize_predictions.py',
        '--gt-json', split_json,
        '--image-dir', split_images,
        '--predictions', predictions,
        '--tier', '2',
        '--limit', '20',
        '--output-dir', os.path.join(out_dir, 'overlays'),
    ], check=True)

print(open(os.path.join(EVAL_ROOT, 'validation', 'tier_metrics.md')).read())

## Cell 8 -- Phase 4 robustness and low-resource experiments

`tools/degrade_images.py` writes blurred / JPEG-compressed / downscaled copies
of the evaluation split, then `tools/run_experiments.py` sweeps every Phase 4
condition and collects one comparable table:

* **diffusion steps** -- `SAMPLE_STEP` 1, 2, 4, 8: accuracy against compute.
  Worth knowing that steps > 1 only run at all because of the `ddim_sample` fix
  in `hierarchialdet/detector.py`; the released code raises `AttributeError`
  there, so this experiment was impossible before it.
* **image degradation** -- how far detection holds up on lower-quality films.
* **clean vs stress subsets** -- the Phase 1 subsets (zero-annotation images,
  high-density films, images with three or more distinct diagnoses).
* **device** -- CPU against GPU on a small slice, for runtime and for a check
  that detections are equivalent.
* **prior-noise** -- corrupt the tier-1 boxes fed into the tier-2 model
  (`NOISY_BOX_INFER_JITTER` / `NOISY_BOX_INFER_DROP`) to test whether the
  hierarchical inference order tolerates imperfect early tiers. This uses the
  opt-in inference-time injection added for this reproduction; the released
  code only injects prior-tier boxes during training.

`tools/benchmark_resources.py` then records parameter count, checkpoint size,
peak GPU memory and throughput -- the numbers a smaller clinic would need to
judge whether this is deployable on their hardware.

In [ ]:
EXPERIMENT_ROOT = '/kaggle/working/experiments'
DEGRADED_ROOT = os.path.join(EXPERIMENT_ROOT, 'degraded')
os.makedirs(EXPERIMENT_ROOT, exist_ok=True)

EXP_JSON, EXP_IMAGES = VALIDATION_TRIPLE_PATH, VALIDATION_IMAGE_DIR

subprocess.run([
    sys.executable, 'tools/degrade_images.py',
    '--json', EXP_JSON,
    '--image-dir', EXP_IMAGES,
    '--output-root', DEGRADED_ROOT,
], check=True)

# Tier-1 predictions over the same split, so the prior-noise experiment has
# real prior-tier boxes to corrupt rather than synthetic ones.
PRIOR_PREDICTIONS = os.path.join(EXPERIMENT_ROOT, 'tier1_predictions.json')
subprocess.run([
    sys.executable, 'tools/dump_predictions.py',
    '--weights', os.path.join(CURRICULUM_ROOT, 'stage1_enumeration', 'model_final.pth'),
    '--json', EXP_JSON, '--image-dir', EXP_IMAGES,
    '--tier', '1', '--output', PRIOR_PREDICTIONS,
], env=dict(os.environ, TIER='1'), check=True)

subprocess.run([
    sys.executable, 'tools/run_experiments.py',
    '--weights', FINAL_WEIGHTS,
    '--json', EXP_JSON,
    '--image-dir', EXP_IMAGES,
    '--output-root', EXPERIMENT_ROOT,
    '--tier', '2',
    '--degraded-root', DEGRADED_ROOT,
    '--prior-predictions', PRIOR_PREDICTIONS,
    '--cpu-limit', '5',
], check=True)

subprocess.run([
    sys.executable, 'tools/benchmark_resources.py',
    '--weights', FINAL_WEIGHTS,
    '--json', EXP_JSON,
    '--image-dir', EXP_IMAGES,
    '--output', os.path.join(EXPERIMENT_ROOT, 'resources.json'),
], check=True)

print(open(os.path.join(EXPERIMENT_ROOT, 'experiment_results.md')).read())

## Cell 9 -- baselines (RetinaNet, Faster R-CNN, DETR)

The paper's central claim is that HierarchicalDet beats these. Each baseline
trains on a single flat tier -- the diagnosis tier by default, which is the only
fully-annotated one and therefore all a conventional detector can use -- and is
scored by the same stock COCO evaluator as the HierarchicalDet dumps, so the
comparison does not run through either model's own evaluation code.

Two honest caveats to carry into the report: DETR comes from Hugging Face
`transformers` rather than from the authors' setup (no DETR config was
released), and each baseline is trained under this reproduction's compute
budget, not the paper's. Both belong in the results table as stated conditions.

The fourth comparison, DiffusionDet, is this same architecture trained without
the hierarchy: `tools/run_curriculum.py --mode single-tier`. That is the
ablation that isolates the noisy box manipulation, since it changes only the
curriculum and holds the architecture fixed.

In [ ]:
BASELINE_ROOT = '/kaggle/working/baselines'
BASELINE_ITER = 200  # <-- smoke value; raise for a real comparison run

for model_name in ('retinanet', 'faster_rcnn'):
    out_dir = os.path.join(BASELINE_ROOT, model_name)
    subprocess.run([
        sys.executable, 'tools/baselines/train_baseline.py',
        '--model', model_name,
        '--train-json', TRAIN_JSON, '--train-images', TRAIN_IMG_DIR,
        '--test-json', VALIDATION_TRIPLE_PATH, '--test-images', VALIDATION_IMAGE_DIR,
        '--output-dir', out_dir,
        '--tier', '2',
        '--max-iter', str(BASELINE_ITER),
    ], check=True)
    subprocess.run([
        sys.executable, 'tools/coco_eval_standalone.py',
        '--gt-json', VALIDATION_TRIPLE_PATH,
        '--predictions', os.path.join(out_dir, 'predictions.json'),
        '--tier', '2',
        '--output', os.path.join(out_dir, 'metrics.json'),
    ], check=True)

detr_dir = os.path.join(BASELINE_ROOT, 'detr')
subprocess.run([
    sys.executable, 'tools/baselines/train_detr.py',
    '--train-json', TRAIN_JSON, '--train-images', TRAIN_IMG_DIR,
    '--test-json', VALIDATION_TRIPLE_PATH, '--test-images', VALIDATION_IMAGE_DIR,
    '--output-dir', detr_dir,
    '--tier', '2',
    '--epochs', '1',  # <-- raise for a real comparison run
], check=True)
subprocess.run([
    sys.executable, 'tools/coco_eval_standalone.py',
    '--gt-json', VALIDATION_TRIPLE_PATH,
    '--predictions', os.path.join(detr_dir, 'predictions.json'),
    '--tier', '2',
    '--output', os.path.join(detr_dir, 'metrics.json'),
], check=True)

# The non-hierarchical DiffusionDet ablation: same model, no curriculum.
subprocess.run([
    sys.executable, 'tools/run_curriculum.py',
    '--mode', 'single-tier',
    '--data-root', os.path.join(EXTRACT_DIR, 'training_data'),
    '--normalized-dir', NORMALIZED_DIR,
    '--val-json', VALIDATION_TRIPLE_PATH, '--val-images', VALIDATION_IMAGE_DIR,
    '--output-root', os.path.join(BASELINE_ROOT, 'diffusiondet_single_tier'),
    '--init-weights', WEIGHTS_PATH,
    '--ims-per-batch', '1',
    '--max-iter', str(MAX_ITER),
    '--eval-period', str(EVAL_PERIOD),
], check=True)

## Cell 10 -- results table and reproducibility checklist

The comparison table and the checklist that has to ship with the report:
environment, package versions, git commit, dataset fingerprints, checkpoint
hashes, seeds, and every documented deviation from the official README.

`--paper` is optional and deliberately not filled in here: someone has to read
the reported numbers out of the paper into a small JSON file, because
hardcoding remembered reference numbers into tooling is exactly how a wrong
baseline ends up in a reproduction report.

In [ ]:
REPORT_ROOT = '/kaggle/working/report'
os.makedirs(REPORT_ROOT, exist_ok=True)

subprocess.run([
    sys.executable, 'tools/make_results_table.py',
    '--hierarchical', os.path.join(EVAL_ROOT, 'validation', 'tier_metrics.json'),
    '--hierarchical-standalone', os.path.join(EVAL_ROOT, 'validation', 'standalone_metrics.json'),
    '--baseline', 'RetinaNet={}'.format(os.path.join(BASELINE_ROOT, 'retinanet', 'metrics.json')),
    '--baseline', 'FasterRCNN={}'.format(os.path.join(BASELINE_ROOT, 'faster_rcnn', 'metrics.json')),
    '--baseline', 'DETR={}'.format(os.path.join(BASELINE_ROOT, 'detr', 'metrics.json')),
    '--output', os.path.join(REPORT_ROOT, 'results_table.md'),
], check=True)

subprocess.run([
    sys.executable, 'tools/make_repro_manifest.py',
    '--output', os.path.join(REPORT_ROOT, 'reproducibility.json'),
    '--curriculum-manifest', os.path.join(CURRICULUM_ROOT, 'curriculum_manifest.json'),
    '--checkpoints', FINAL_WEIGHTS, WEIGHTS_PATH,
    '--datasets', QUADRANT_TRAIN_JSON, QUADRANT_ENUM_TRAIN_JSON, DISEASE_JSON,
    VALIDATION_TRIPLE_PATH, TEST_COCO_JSON,
    '--configs', 'configs/diffdet.custom.swinbase.nonpretrain.yaml', 'configs/Base-DiffusionDet.yaml',
], check=True)

print(open(os.path.join(REPORT_ROOT, 'results_table.md')).read())